# 02 · Bronze

Landing CSV → **Delta table** at `data/bronze/enem_raw`, partitioned by `NU_ANO`.

All columns are kept **as strings** on purpose: Bronze is a faithful copy of the source,
so a malformed value never breaks ingestion. Typing happens in Silver.
Two audit columns are added: `_ingestion_timestamp` and `_source_file`.

In [2]:
%run ./00_setup.ipynb

:: loading settings :: url = jar:file:/home/gabriel/Documents/GitHub/ETL-com-PySpark/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/gabriel/Documents/GitHub/ETL-com-PySpark/.ivy2/cache
The jars for the packages stored in: /home/gabriel/Documents/GitHub/ETL-com-PySpark/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-25767be3-7917-4f67-8002-6c555b0016ab;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 100ms :: artifacts dl 4ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	------------------------------------

Spark 3.5.3 ready


In [3]:
from pyspark.sql.functions import col, current_timestamp

df_raw = (
    spark.read.option("header", "true")
    .option("sep", ";")
    .option("encoding", "ISO-8859-1")
    .csv(str(LANDING_CSV))
)

df_bronze = (
    df_raw.withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))  # Spark's hidden file metadata column
)

In [4]:
df_bronze.write.format("delta").mode("overwrite").partitionBy("NU_ANO").save(BRONZE_TABLE)

26/11/25 14:19:53 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Read the table back from storage.

In [5]:
bronze = spark.read.format("delta").load(BRONZE_TABLE)
print(f"Bronze: {bronze.count()} rows | {len(bronze.columns)} columns")
bronze.select("NU_INSCRICAO", "NU_ANO", "SG_UF_PROVA", "NU_NOTA_MT", "_ingestion_timestamp", "_source_file").show(5, truncate=False)

Bronze: 20 rows | 62 columns
+------------+------+-----------+----------+--------------------------+---------------------------------------------------------------------------------------------------+
|NU_INSCRICAO|NU_ANO|SG_UF_PROVA|NU_NOTA_MT|_ingestion_timestamp      |_source_file                                                                                       |
+------------+------+-----------+----------+--------------------------+---------------------------------------------------------------------------------------------------+
|200001000001|2020  |SP         |640.2     |2026-11-25 14:19:54.068648|file:/home/gabriel/Documents/GitHub/ETL-com-PySpark/data/landing/enem_2020/MICRODADOS_ENEM_2020.csv|
|200001000002|2020  |RJ         |530.7     |2026-11-25 14:19:54.068648|file:/home/gabriel/Documents/GitHub/ETL-com-PySpark/data/landing/enem_2020/MICRODADOS_ENEM_2020.csv|
|200001000003|2020  |MG         |740.5     |2026-11-25 14:19:54.068648|file:/home/gabriel/Documents/GitHub/ETL-

Every write is a commit in the Delta transaction log (`_delta_log/`).

In [6]:
spark.sql(f"DESCRIBE HISTORY delta.`{BRONZE_TABLE}`").select("version", "timestamp", "operation", "operationMetrics").show(truncate=False)

+-------+-----------------------+---------+-------------------------------------------------------------+
|version|timestamp              |operation|operationMetrics                                             |
+-------+-----------------------+---------+-------------------------------------------------------------+
|0      |2026-11-25 14:19:55.858|WRITE    |{numFiles -> 1, numOutputRows -> 20, numOutputBytes -> 20285}|
+-------+-----------------------+---------+-------------------------------------------------------------+



In [7]:
spark.stop()